# Assignment 7: Full Round 2-Style Problem — Retentive Attention (100 points)

## Problem Description

In this problem, you will implement **Retentive Self-Attention (RetNet)**, a recently proposed alternative to standard Transformer attention that enables efficient recurrent inference while maintaining parallel training.

### Background: Retention Mechanism

Standard attention computes: $\text{Attn}(Q, K, V) = \text{softmax}(QK^T / \sqrt{d})V$

The retention mechanism replaces softmax attention with an **exponentially decaying** attention pattern. For a sequence of length $L$ with head dimension $d$:

**Parallel form (for training):**

Given input $X \in \mathbb{R}^{B \times L \times d_{\text{model}}}$:

1. Project: $Q = (XW_Q) \odot \Theta$, $K = (XW_K) \odot \bar{\Theta}$, $V = XW_V$
   where $W_Q, W_K \in \mathbb{R}^{d_{\text{model}} \times d}$, $W_V \in \mathbb{R}^{d_{\text{model}} \times d}$,
   $\Theta$ is the complex rotation matrix (we will use a simplified real version), and $\bar{\Theta}$ is its conjugate.

2. **Simplified (real-valued) version:** Instead of complex rotations, use position-dependent scaling:
   $$\Theta_n = e^{in\theta}$$
   For the real-valued simplification, apply RoPE-style rotations (which you may implement as simple element-wise scaling for this problem).

3. Compute the **decay matrix** $D \in \mathbb{R}^{L \times L}$:
   $$D_{nm} = \begin{cases} \gamma^{n-m} & \text{if } n \geq m \\ 0 & \text{if } n < m \end{cases}$$
   where $\gamma \in (0, 1)$ is the decay factor (e.g., $\gamma = 0.95$). This is a **causal** lower-triangular matrix with exponential decay.

4. Retention output:
   $$\text{Retention}(X) = (QK^T \odot D) V$$

   Note: No softmax! The decay matrix $D$ replaces softmax's role in controlling attention distribution.

**Multi-Scale Retention:**
- Use $h$ heads with different decay rates: $\gamma_1, \gamma_2, \ldots, \gamma_h$
- Concatenate head outputs and project: $\text{MSR}(X) = (\text{Concat}[\text{head}_1, \ldots, \text{head}_h]) W_O$
- Apply GroupNorm to each head's output before concatenation

**Recurrent form (for inference):**

The state $s_n \in \mathbb{R}^{d \times d}$ is updated as:
$$s_n = \gamma \cdot s_{n-1} + k_n^T v_n$$
$$o_n = q_n \cdot s_n$$

This enables $O(1)$ per-step inference (like an RNN) while training in parallel (like a Transformer).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import math
import numpy as np

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

## Part 1: Decay Matrix Properties (6 points)

**[Non-coding]**

1. (2 points) Write out the $4 \times 4$ decay matrix $D$ for $\gamma = 0.9$. What is $D_{3,0}$?

2. (2 points) The decay matrix is causal (lower triangular). How does this compare to the causal mask in standard Transformer attention? What role does it serve?

3. (2 points) What happens to the retention mechanism as $\gamma \to 1$? As $\gamma \to 0$?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2: Construct the Decay Matrix (8 points)

**[Coding]** Implement a function that constructs the decay matrix $D$.

$$D_{nm} = \begin{cases} \gamma^{n-m} & \text{if } n \geq m \\ 0 & \text{if } n < m \end{cases}$$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def build_decay_matrix(L, gamma):
    """
    Build the L x L causal decay matrix.
    
    Args:
        L: sequence length
        gamma: decay factor in (0, 1)
    
    Returns:
        D: (L, L) tensor, lower-triangular with exponential decay
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: Single-Head Retention (12 points)

**[Coding]** Implement single-head retention (parallel form).

$$\text{Retention}(X) = (QK^T \odot D)V$$

where $Q = XW_Q$, $K = XW_K$, $V = XW_V$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SingleHeadRetention(nn.Module):
    def __init__(self, d_model, d_head, gamma=0.95):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        # x: (B, L, d_model)
        # Returns: (B, L, d_head)
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 4: Shape Analysis (6 points)

**[Non-coding]** For $B = 2$, $L = 32$, $d_{\text{model}} = 64$, $d_{\text{head}} = 16$:

1. (2 points) What is the shape of $QK^T$?
2. (2 points) What is the shape of $(QK^T \odot D)$? What broadcasting is needed for $D$?
3. (2 points) What is the total computation complexity of single-head retention? Compare with standard attention of the same dimensions.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 5: Multi-Scale Retention (12 points)

**[Coding]** Implement Multi-Scale Retention with $h$ heads.

- Each head has a different decay rate: $\gamma_i = 1 - 2^{-(5 + i)}$ for $i = 0, 1, \ldots, h-1$
- Apply GroupNorm (groups=h) to the concatenated head outputs before the output projection
- Head dimension: $d_{\text{head}} = d_{\text{model}} / h$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MultiScaleRetention(nn.Module):
    def __init__(self, d_model, num_heads=4):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        # x: (B, L, d_model)
        # Returns: (B, L, d_model)
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 6: Smoke Test (5 points)

**[Coding]** Verify that your Multi-Scale Retention produces the correct output shape.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Smoke test: B=2, L=32, d_model=64, num_heads=4

""" END OF THIS PART """

## Part 7: Recurrent Form (12 points)

**[Coding]** Implement the recurrent form for efficient inference.

$$s_n = \gamma \cdot s_{n-1} + k_n^T v_n$$
$$o_n = q_n \cdot s_n$$

Process the sequence one step at a time. The state $s$ has shape $(d_{\text{head}}, d_{\text{head}})$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class RecurrentRetention(nn.Module):
    def __init__(self, d_model, d_head, gamma=0.95):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        """Process sequence step by step (recurrent form)."""
        # x: (B, L, d_model)
        # Returns: (B, L, d_head)
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 8: Verify Equivalence (8 points)

**[Coding]** Show that the parallel and recurrent forms produce the same output (up to numerical precision).

1. Create a `SingleHeadRetention` and a `RecurrentRetention` with the **same weights**.
2. Pass the same input through both.
3. Verify that the outputs are close (max absolute difference < 1e-5).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Verify equivalence of parallel and recurrent forms

""" END OF THIS PART """

## Part 9: RetNet Block (8 points)

**[Coding]** Build a full RetNet block.

- Multi-Scale Retention (pre-norm)
- FFN: Linear(d, 2d) → GELU → Linear(2d, d) (pre-norm)
- Residual connections
- LayerNorm

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class RetNetBlock(nn.Module):
    def __init__(self, d_model, num_heads=4):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 10: Language Model (8 points)

**[Coding]** Build a causal language model using RetNet blocks.

- Token embedding + positional embedding
- 3 RetNet blocks
- LM head: LayerNorm → Linear(d_model, vocab_size)
- vocab_size=256, d_model=64, num_heads=4, max_len=128

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class RetNetLM(nn.Module):
    def __init__(self, vocab_size=256, d_model=64, num_heads=4,
                 num_layers=3, max_len=128):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        # x: (B, L) token IDs
        # Returns: (B, L, vocab_size) logits
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 11: Train on Synthetic Data (8 points)

**[Coding]** Train the RetNet LM on a simple pattern-copying task.

Generate data: sequences of 32 random bytes repeated twice (i.e., if the first 16 bytes are `[a, b, c, ...]`, the full sequence is `[a, b, c, ..., a, b, c, ...]`). The model should learn to predict the second half from the first.

- 500 training sequences
- Train 20 epochs, Adam lr=1e-3, batch_size=32
- Cross-entropy loss on next-token prediction

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Generate data and train

""" END OF THIS PART """

## Part 12: Inference Comparison (7 points)

**[Non-coding + Coding]**

1. (3 points) **[Non-coding]** Compare the inference complexity per step of RetNet (recurrent form) vs. standard Transformer. What is the time and memory complexity for generating the $n$-th token?

2. (4 points) **[Coding]** Time the generation of 100 tokens using both the parallel form (recomputing for each new token) and the recurrent form. Print the wall-clock times.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Inference timing comparison

""" END OF THIS PART """

## Part 13: Limitations (5 points)

**[Non-coding]**

1. (2 points) The retention mechanism does not use softmax. What mathematical property of softmax attention is lost? How might this affect the model's behavior?

2. (3 points) The exponential decay means that information from distant tokens is heavily discounted. For what types of tasks would this be (a) advantageous and (b) disadvantageous?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 14: Extension — Chunk-Wise Form (5 points)

**[Non-coding]** RetNet has a third form: chunk-wise, which processes the sequence in fixed-size chunks to balance parallelism and memory.

1. (2 points) If the chunk size is $C$, what is the memory complexity per chunk? How does this compare to the parallel form over the full sequence?

2. (3 points) Describe the algorithm for chunk-wise retention: how would you process a sequence of length $L$ in chunks of size $C$? What state is passed between chunks?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """